# Gradient Boosting - Temporal Validation and Drift Analysis

The previous notebook explored selected Gradient Boosting hyperparameters manually on the 2018 validation set. Increasing tree depth produced the largest improvement, but repeatedly making decisions on a single validation year risks overfitting to that period.

This notebook evaluates Gradient Boosting using expanding-window temporal cross-validation within the 2010–2017 development period. The goals are to:

- assess how stable model performance is across years,
- diagnose periods in which the model fails and relate them to changes in the data,
- compare tree depths under time-aware validation.

The 2018 validation set is not used in this notebook, and 2019 remains completely unseen for final model evaluation.

## Experimental Design

The available labeled transaction history is divided according to time:

- **2010–2017:** model development and hyperparameter tuning
- **2018:** validation and model comparison
- **2019:** final out-of-time test set

Hyperparameter tuning must use only information available within the 2010–2017 development period. The temporal order of transactions will be preserved during cross-validation to avoid training models on future observations and evaluating them on the past.

In [60]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score

from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
)

from finsight.fraud_preprocessing import (
    encode_categorical_features,
    handle_missing_values,
)

from finsight.database import (
    connect_to_database,
    test_database_connection,
)

from finsight.fraud_data import (
    BASE_FEATURES,
    download_yearly_data,
    download_full_year_data,
)

In [2]:
from finsight.utils.notebook_notifications import enable_cell_notifications

enable_cell_notifications()

In [3]:
engine = connect_to_database()
test_database_connection(engine)

Connected!


## Development Data

Hyperparameter tuning is performed exclusively on data from 2010–2017.

The development dataset is loaded separately for each year to preserve temporal boundaries required for time-aware cross-validation. Non-fraud transactions are sampled proportionally to the original yearly transaction volumes, with a total sampling budget of 300,000 observations.

The previously selected base feature set is retained so that the tuning experiments isolate the effect of model hyperparameters rather than changes in feature representation.

In [4]:
features = BASE_FEATURES

yearly_data = download_yearly_data(
    engine,
    start_year=2010,
    end_year=2017,
    total_non_fraud_limit=300000,
    features=features
)

In [5]:
for year, data in yearly_data.items():
    print(
        year,
        data["X"].shape,
        data["y"].value_counts().to_dict()
    )

2010 (37148, 13) {False: 34575, True: 2573}
2011 (36048, 13) {False: 36011, True: 37}
2012 (37815, 13) {False: 36892, True: 923}
2013 (39124, 13) {False: 37787, True: 1337}
2014 (38803, 13) {False: 38139, True: 664}
2015 (40896, 13) {False: 38707, True: 2189}
2016 (41251, 13) {False: 38803, True: 2448}
2017 (39258, 13) {False: 39086, True: 172}


## Temporal Development Set

The yearly datasets are concatenated in chronological order from 2010 to 2017.

The feature matrix and target vector are kept separate, while an additional year vector is created to identify the temporal origin of each observation. The year information is used only for constructing temporal validation folds and is not included as a model feature.

In [6]:
X_development = pd.concat(
    [yearly_data[year]["X"] for year in range(2010, 2018)],
    ignore_index=True
)
X_development.shape

(310343, 13)

In [7]:
y_development = pd.concat(
    [yearly_data[year]["y"] for year in range(2010, 2018)],
    ignore_index=True
)

y_development.shape

(310343,)

In [8]:
development_years = []

for year in yearly_data:
    development_years.extend(
        [year] * len(yearly_data[year]["X"])
    )

development_years = pd.Series(development_years)

development_years.shape

(310343,)

In [9]:
development_years.value_counts().sort_index()

2010    37148
2011    36048
2012    37815
2013    39124
2014    38803
2015    40896
2016    41251
2017    39258
Name: count, dtype: int64

## Temporal Cross-Validation

In [10]:
train_mask = development_years.isin([2010, 2011, 2012])
train_indices = development_years[train_mask].index
train_indices

RangeIndex(start=0, stop=111011, step=1)

In [11]:
development_years[train_mask].unique()

array([2010, 2011, 2012])

In [12]:
val_mask = development_years.isin([2013])
val_indices = development_years[val_mask].index
val_indices

RangeIndex(start=111011, stop=150135, step=1)

In [13]:
development_years[val_mask].unique()

array([2013])

In [14]:
train_indices.intersection(val_indices)

RangeIndex(start=0, stop=0, step=1)

### Automated Temporal Folds

In [15]:
def create_temporal_folds(development_years, min_years=3):
    folds = []
    years = development_years.unique()

    for val_year in years[min_years:]:
        val_mask = (development_years == val_year)
        val_indices = development_years[val_mask].index

        train_mask = development_years < val_year
        train_indices = development_years[train_mask].index

        folds.append((train_indices, val_indices))
        
    return folds

In [16]:
folds = create_temporal_folds(development_years)
len(folds)

5

In [17]:
for train_indices, val_indices in folds:
    print(development_years[train_indices].unique(), development_years[val_indices].unique())

[2010 2011 2012] [2013]
[2010 2011 2012 2013] [2014]
[2010 2011 2012 2013 2014] [2015]
[2010 2011 2012 2013 2014 2015] [2016]
[2010 2011 2012 2013 2014 2015 2016] [2017]


### Expanding Window Validation

To preserve the temporal structure of the transaction data, model configurations are evaluated using expanding-window cross-validation rather than random cross-validation.

Each fold trains the model using all available historical years and validates it on the immediately following year:

- Fold 1: 2010–2012 → 2013
- Fold 2: 2010–2013 → 2014
- Fold 3: 2010–2014 → 2015
- Fold 4: 2010–2015 → 2016
- Fold 5: 2010–2016 → 2017

This setup simulates repeated model development over time while preventing future observations from being used to predict the past.

The folds store only the training and validation indices. Training data for each fold is selected from the undersampled development dataset, while validation is performed on the complete data for the validation year, as described below.

## Validation Folds at Natural Fraud Rate

The development dataset is undersampled: it contains all fraudulent transactions but only a sample of legitimate ones. Evaluating validation folds on this sample would artificially inflate the fraud rate (e.g. 5.35% instead of approximately 0.24% in 2015) and, as a result, overstate precision and F1-score.

Therefore, models are trained on the undersampled historical years, but each validation fold is evaluated on the complete data for its year, preserving the natural fraud rate. This makes fold results directly comparable with the 2018 validation set.

In [18]:
validation_years = [2013, 2014, 2015, 2016, 2017]

full_year_data = {}

for year in validation_years:
    X_year, y_year = download_full_year_data(engine, year, features=features)

    full_year_data[year] = {
        "X": X_year,
        "y": y_year
    }

for year, data in full_year_data.items():
    print(
        year,
        data["X"].shape,
        data["y"].value_counts().to_dict(),
        f"fraud rate: {data['y'].mean() * 100:.4f}%"
    )

2013 (907304, 13) {False: 905967, True: 1337} fraud rate: 0.1474%
2014 (915073, 13) {False: 914409, True: 664} fraud rate: 0.0726%
2015 (930224, 13) {False: 928035, True: 2189} fraud rate: 0.2353%
2016 (932762, 13) {False: 930314, True: 2448} fraud rate: 0.2624%
2017 (937284, 13) {False: 937112, True: 172} fraud rate: 0.0184%


### Fold Preparation

Each fold uses the undersampled development data from all previous years for training and the complete data from the validation year for evaluation. The same preprocessing steps are applied in every fold, with the encoder and imputation medians fitted on the training data only.

In [19]:
def prepare_fold(train_indices, val_year):
    X_train = X_development.iloc[train_indices].copy()
    y_train = y_development.iloc[train_indices]

    X_val = full_year_data[val_year]["X"].copy()
    y_val = full_year_data[val_year]["y"]

    X_train["mcc_key"] = X_train["mcc_key"].astype(str)
    X_val["mcc_key"] = X_val["mcc_key"].astype(str)

    X_train_encoded, X_val_encoded = encode_categorical_features(X_train, X_val)
    X_train_encoded, X_val_encoded = handle_missing_values(X_train_encoded, X_val_encoded)

    return X_train_encoded, y_train, X_val_encoded, y_val

## Temporal Baseline Evaluation

Before starting systematic hyperparameter tuning, the previously selected Gradient Boosting configuration (`max_depth=6`) is evaluated across the temporal folds.

The objective of this step is to establish a temporal baseline and verify that model performance is sufficiently stable across validation periods before comparing additional hyperparameter configurations.

In [20]:
fold_results = []
fold_predictions = {}

for (train_indices, _), year in zip(folds, validation_years):
    X_train, y_train, X_val, y_val = prepare_fold(train_indices, year)

    gb = GradientBoostingClassifier(max_depth=6, random_state=42)
    gb.fit(X_train, y_train)

    y_pred = gb.predict(X_val)
    fold_predictions[year] = y_pred

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    fold_results.append({
        "year": year,
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

fold_results = pd.DataFrame(fold_results)

fold_results[["year", "f1"]]

,year,f1
0,2013,0.519294
1,2014,0.373383
2,2015,0.058558
3,2016,0.624874
4,2017,0.004570


## Temporal Validation Diagnostics

The temporal baseline revealed substantial variation in model performance across validation periods. The model achieved its strongest F1-scores in 2016 (0.625) and 2013 (0.519), a moderate result in 2014 (0.373), while performance collapsed in 2015 (0.059) and 2017 (0.005).

Before proceeding with hyperparameter tuning, these periods are investigated to determine whether the instability is associated with changes in class prevalence, transaction characteristics, or fraud patterns over time.

The analysis focuses particularly on two distinct failure modes:

- **2015:** reduced precision and a large increase in false positives.
- **2017:** extremely low recall and failure to identify most fraudulent transactions.

In [21]:
fraud_by_year = pd.DataFrame([
    {
        "year": year,
        "transactions": len(data["y"]),
        "frauds": int(data["y"].sum()),
        "fraud_rate": data["y"].mean() * 100
    }
    for year, data in full_year_data.items()
]).set_index("year")

fraud_by_year

,transactions,frauds,fraud_rate
year,,,
2013,907304,1337,0.147360
2014,915073,664,0.072563
2015,930224,2189,0.235320
2016,932762,2448,0.262446
2017,937284,172,0.018351


### Model Performance Across Validation Periods

Fraud prevalence varies substantially over time, from 0.018% of transactions in 2017 to 0.262% in 2016. However, changes in class prevalence alone do not explain the differences in model performance.

To understand how the model fails in each validation period, precision, recall and confusion-matrix components are examined separately.

In [22]:
fold_results

,year,precision,recall,f1,tn,fp,fn,tp
0,2013,0.432702,0.649215,0.519294,904829,1138,469,868
1,2014,0.247428,0.760542,0.373383,912873,1536,159,505
2,2015,0.030495,0.734125,0.058558,876945,51090,582,1607
3,2016,0.531842,0.757353,0.624874,928682,1632,594,1854
4,2017,0.002480,0.029070,0.004570,935101,2011,167,5


In [23]:
fold_results["actual_fraud_rate"] = (
    (fold_results["tp"] + fold_results["fn"]) /
    (fold_results["tn"] + fold_results["fp"] + fold_results["fn"] + fold_results["tp"])
) * 100

fold_results["predicted_fraud_rate"] = (
    (fold_results["tp"] + fold_results["fp"]) /
    (fold_results["tn"] + fold_results["fp"] + fold_results["fn"] + fold_results["tp"])
) * 100

fold_results[
    ["year", "actual_fraud_rate", "predicted_fraud_rate"]
]

,year,actual_fraud_rate,predicted_fraud_rate
0,2013,0.147360,0.221095
1,2014,0.072563,0.223042
2,2015,0.235320,5.664980
3,2016,0.262446,0.373729
4,2017,0.018351,0.215090


### Temporal Changes in Transaction Characteristics

The error analysis indicates that the weaker validation periods exhibit different failure modes. Before investigating 2015 and 2017 individually, transaction characteristics are compared across years for both fraudulent and legitimate observations.

The analysis considers:

- transaction amount relative to historical user and card behavior,
- transaction channel (`use_chip`),
- whether observed changes affect fraud, legitimate transactions, or both.

These comparisons provide context for the period-specific diagnostics that follow.

In [24]:
fraud_features = X_development.copy()

fraud_features["year"] = development_years.values
fraud_features["is_fraud"] = y_development.values

fraud_features = fraud_features[fraud_features["is_fraud"] == True]

fraud_feature_medians = fraud_features.groupby("year")[
    [
        "amount",
        "user_previous_avg_amount",
        "amount_vs_user_avg",
        "card_previous_avg_amount",
        "amount_vs_card_avg"
    ]
].median()

fraud_feature_medians

,amount,user_previous_avg_amount,amount_vs_user_avg,card_previous_avg_amount,amount_vs_card_avg
year,,,,,
2010,74.290,43.540,28.830,44.160,29.180
2011,99.890,59.670,40.550,58.000,40.720
2012,47.710,41.000,4.750,38.990,7.010
2013,86.110,41.500,43.900,41.170,47.230
2014,87.495,41.930,40.560,38.795,44.515
2015,88.790,41.070,45.730,39.340,46.800
2016,84.665,40.695,43.540,39.700,44.095
2017,29.495,43.460,-10.925,37.610,-7.270


In [25]:
nonfraud_features = X_development.copy()

nonfraud_features["year"] = development_years.values
nonfraud_features["is_fraud"] = y_development.values

nonfraud_features = nonfraud_features[
    nonfraud_features["is_fraud"] == False
]

nonfraud_feature_medians = nonfraud_features.groupby("year")[
    [
        "amount",
        "user_previous_avg_amount",
        "amount_vs_user_avg",
        "card_previous_avg_amount",
        "amount_vs_card_avg"
    ]
].median()

nonfraud_feature_medians

,amount,user_previous_avg_amount,amount_vs_user_avg,card_previous_avg_amount,amount_vs_card_avg
year,,,,,
2010,29.72,40.300,-9.85,39.29,-8.190
2011,29.24,40.330,-10.67,39.44,-8.740
2012,28.97,40.245,-10.43,39.00,-8.520
2013,29.36,40.440,-10.53,39.19,-8.680
2014,28.98,40.150,-10.56,38.70,-8.450
2015,28.84,39.960,-10.73,38.64,-8.770
2016,28.93,40.000,-10.85,38.72,-9.025
2017,28.42,39.880,-10.87,38.43,-8.950


In [26]:
fraud_use_chip_by_year = pd.crosstab(
    fraud_features["year"],
    fraud_features["use_chip"],
    normalize="index"
) * 100

fraud_use_chip_by_year

use_chip,Chip Transaction,Online Transaction,Swipe Transaction
year,,,
2010,0.000000,90.050525,9.949475
2011,0.000000,94.594595,5.405405
2012,0.000000,75.081257,24.918743
2013,0.000000,84.293194,15.706806
2014,0.000000,84.638554,15.361446
2015,8.725445,85.153038,6.121517
2016,7.761438,85.620915,6.617647
2017,98.837209,0.000000,1.162791


In [27]:
nonfraud_use_chip_by_year = pd.crosstab(
    nonfraud_features["year"],
    nonfraud_features["use_chip"],
    normalize="index"
) * 100

nonfraud_use_chip_by_year

use_chip,Chip Transaction,Online Transaction,Swipe Transaction
year,,,
2010,0.000000,10.672451,89.327549
2011,0.000000,10.827247,89.172753
2012,0.000000,11.034913,88.965087
2013,0.000000,11.490724,88.509276
2014,0.000000,11.610163,88.389837
2015,70.651303,12.318185,17.030511
2016,71.115635,12.140814,16.743551
2017,70.861690,12.127104,17.011206


### 2015 Performance Degradation

The 2015 validation period exhibits a primarily false-positive-driven failure mode.

Although the model retains relatively strong fraud detection ability (**recall = 0.734**), precision collapses to **0.030**, resulting in an F1-score of **0.059**. The model incorrectly classifies **51,090 legitimate transactions as fraud** and flags **5.66%** of all transactions, compared with an actual fraud rate of only **0.24%**.

The following analysis investigates whether these false positives are associated with temporal changes in the transaction environment.

#### Recreating the 2015 Validation Fold

The 2015 validation data and the corresponding predictions from the temporal baseline evaluation are retrieved. This allows individual prediction errors to be inspected directly without retraining the model.

In [28]:
X_val_2015 = full_year_data[2015]["X"]
y_val_2015 = full_year_data[2015]["y"]

y_pred_2015 = fold_predictions[2015]

#### False Positive Analysis

Because the 2015 degradation is primarily driven by reduced precision, the analysis focuses on legitimate transactions incorrectly classified as fraud.

First, the transaction-channel composition of the false positives is examined.

In [29]:
false_positive_mask_2015 = (
    (y_val_2015.to_numpy() == False) &
    (y_pred_2015 == True)
)

false_positives_2015 = X_val_2015.loc[false_positive_mask_2015]

false_positives_2015["use_chip"].value_counts(normalize=True) * 100

use_chip
Chip Transaction      98.308867
Online Transaction     1.561950
Swipe Transaction      0.129184
Name: proportion, dtype: float64

Nearly all false positives are Chip Transactions. To determine whether this reflects only the composition of the false-positive group or a broader model problem with legitimate chip transactions, the false-positive rate within that group is calculated next.

In [30]:
nonfraud_chip_mask_2015 = (
    (y_val_2015.to_numpy() == False) &
    (X_val_2015["use_chip"].to_numpy() == "Chip Transaction")
)

nonfraud_chip_predictions_2015 = y_pred_2015[nonfraud_chip_mask_2015]

nonfraud_chip_predictions_2015.mean() * 100

np.float64(7.6276707706186)

#### 2015 Diagnostic Conclusion

The 2015 validation period shows a different failure mode from the severe performance collapse observed in 2017.

Although the model maintained relatively strong fraud detection ability in 2015 (**recall = 0.734**), precision decreased dramatically to **0.030**, resulting in an F1-score of **0.059**. The primary source of this degradation was an extreme increase in false positives: **51,090 legitimate transactions were incorrectly classified as fraud**, compared with between 1,138 and 1,632 in 2013, 2014 and 2016.

Further analysis revealed a major temporal change in the transaction channel distribution. Until 2014, legitimate transactions contained virtually no `Chip Transaction` observations. In 2015, however, **70.65% of legitimate transactions were chip-based**.

The model, trained only on earlier periods, therefore encountered a transaction channel that was largely absent from its historical training data.

This shift was strongly associated with the observed false-positive problem:

- **98.31% of all false positives in 2015 were Chip Transactions**
- **7.63% of legitimate Chip Transactions were incorrectly classified as fraud**
- fraud recall nevertheless remained relatively high at **0.734**

Interestingly, the distribution of legitimate transaction channels remained almost unchanged in 2016, while model performance recovered to **F1 = 0.625**. Under the expanding-window validation scheme, the 2016 training set already included 2015 observations, giving the model exposure to the newly prevalent chip-based transaction environment.

These results suggest that the 2015 performance degradation is primarily associated with a **temporal shift in the transaction environment and an elevated false-positive rate**, rather than a failure to recognize fraudulent transactions themselves.

The evidence demonstrates a strong association rather than establishing that the channel shift alone caused the performance degradation.

### 2017 Performance Collapse

Unlike 2015, the 2017 validation period is characterized by an almost complete failure to identify fraudulent transactions.

The model achieved:

- **precision = 0.002**
- **recall = 0.029**
- **F1 = 0.005**

Only **5 of 172 fraudulent transactions** were correctly identified.

The following analysis investigates whether this failure is associated with a change in the characteristics of fraudulent transactions in 2017.

#### Recreating the 2017 Validation Fold

The 2017 validation data and the corresponding predictions from the temporal baseline evaluation are retrieved, together with the historical training data used for this fold. This allows the fraudulent transactions missed by the model to be inspected directly without retraining the model.

In [31]:
X_train_2017 = X_development[development_years < 2017]

X_val_2017 = full_year_data[2017]["X"]
y_val_2017 = full_year_data[2017]["y"]

y_pred_2017 = fold_predictions[2017]

#### False Negative Analysis

The 2017 performance collapse is driven by extremely low recall. Therefore, the analysis focuses on fraudulent transactions incorrectly classified as legitimate.

The transaction-channel distribution of these false negatives is examined first.

In [32]:
false_negative_mask_2017 = (
    (y_val_2017.to_numpy() == True) &
    (y_pred_2017 == False)
)

false_negative_2017 = X_val_2017.loc[false_negative_mask_2017]

false_negative_2017["use_chip"].value_counts(normalize=True) * 100

use_chip
Chip Transaction     98.802395
Swipe Transaction     1.197605
Name: proportion, dtype: float64

Almost all missed frauds are Chip Transactions. However, because Chip Transactions also account for nearly all fraud in 2017, this alone does not demonstrate that the model specifically fails on this channel.

The detection rate among all fraudulent Chip Transactions is therefore examined next.

In [33]:
fraud_chip_mask_2017 = (
    (y_val_2017.to_numpy() == True) &
    (X_val_2017["use_chip"].to_numpy() == "Chip Transaction")
)

fraud_chip_predictions_2017 = y_pred_2017[fraud_chip_mask_2017]

fraud_chip_predictions_2017.mean() * 100

np.float64(2.941176470588235)

#### Historical vs. 2017 Chip Fraud

The low detection rate for Chip Transactions does not by itself explain the 2017 collapse, because chip-based fraud was also present in earlier training data.

To determine whether the nature of chip-based fraud changed, historical Chip Transactions classified as fraud are separated from Chip Transactions classified as fraud in 2017.

These groups will be compared to determine whether the 2017 fraud pattern differs from the historical chip-fraud observations available to the model.

In [34]:
historical_chip_fraud_mask = (
    (y_development == True) &
    (development_years < 2017) &
    (X_development["use_chip"] == "Chip Transaction")
)

historical_chip_frauds = X_development[historical_chip_fraud_mask]

len(historical_chip_frauds)

381

In [35]:
chip_fraud_2017_mask = (
    (y_development == True) &
    (development_years == 2017) &
    (X_development["use_chip"] == "Chip Transaction")
)

chip_fraud_2017 = X_development[chip_fraud_2017_mask]

len(chip_fraud_2017)

170

In [36]:
historical_chip_frauds_medians = historical_chip_frauds[
    [
        "amount",
        "amount_vs_user_avg",
        "amount_vs_card_avg",
        "user_previous_transaction_count",
        "user_previous_avg_amount",
        "card_previous_transaction_count",
        "card_previous_avg_amount"
    ]
].median()

historical_chip_frauds_medians

amount                               76.17
amount_vs_user_avg                   31.77
amount_vs_card_avg                   34.59
user_previous_transaction_count    6016.00
user_previous_avg_amount             41.05
card_previous_transaction_count    1742.00
card_previous_avg_amount             41.87
dtype: float64

In [37]:
chip_fraud_2017_medians = chip_fraud_2017[
    [
        "amount",
        "amount_vs_user_avg",
        "amount_vs_card_avg",
        "user_previous_transaction_count",
        "user_previous_avg_amount",
        "card_previous_transaction_count",
        "card_previous_avg_amount"
    ]
].median()

chip_fraud_2017_medians

amount                               28.440
amount_vs_user_avg                  -12.015
amount_vs_card_avg                   -7.680
user_previous_transaction_count    7805.000
user_previous_avg_amount             43.455
card_previous_transaction_count    2143.000
card_previous_avg_amount             37.600
dtype: float64

In [38]:
chip_nonfraud_2017_mask = (
    (y_development == False) &
    (development_years == 2017) &
    (X_development["use_chip"] == "Chip Transaction")
)

chip_nonfraud_2017 = X_development[chip_nonfraud_2017_mask]

len(chip_nonfraud_2017)

27697

In [39]:
chip_nonfraud_2017_medians = chip_nonfraud_2017[
    [
        "amount",
        "amount_vs_user_avg",
        "amount_vs_card_avg",
        "user_previous_transaction_count",
        "user_previous_avg_amount",
        "card_previous_transaction_count",
        "card_previous_avg_amount"
    ]
].median()

chip_nonfraud_2017_medians

amount                               26.38
amount_vs_user_avg                  -12.47
amount_vs_card_avg                  -10.16
user_previous_transaction_count    8939.00
user_previous_avg_amount             40.02
card_previous_transaction_count    3130.00
card_previous_avg_amount             38.45
dtype: float64

#### Categorical Feature Comparison

The numerical comparison indicates that fraudulent and legitimate Chip Transactions in 2017 are highly similar across several amount-based behavioral features, particularly `amount_vs_user_avg` and `amount_vs_card_avg`.

To determine whether the two groups can still be distinguished through other transaction characteristics, the analysis is extended to categorical and discrete features.

The following features are examined:

- `mcc_key`
- `merchant_id`
- `transaction_hour`
- `day_of_week`
- `is_weekend`

`use_chip` is excluded because both groups are intentionally restricted to Chip Transactions.

In [40]:
chip_fraud_2017["mcc_key"].value_counts(normalize=True).head(10) * 100

mcc_key
10    19.411765
12    13.529412
11     7.058824
4      5.882353
14     4.705882
35     4.705882
2      3.529412
67     3.529412
68     3.529412
25     3.529412
Name: proportion, dtype: float64

In [41]:
chip_nonfraud_2017["mcc_key"].value_counts(normalize=True).head(10) * 100

mcc_key
4     13.907643
14    12.470665
2     11.957974
1      8.434126
25     6.657761
16     5.137741
9      4.942774
8      4.314547
24     3.953497
10     2.920894
Name: proportion, dtype: float64

In [42]:
historical_chip_frauds["mcc_key"].value_counts(normalize=True).head(10) * 100

mcc_key
16    26.771654
4     12.073491
19     6.036745
12     6.036745
25     4.724409
29     4.461942
35     4.199475
15     3.412073
67     3.412073
3      3.149606
Name: proportion, dtype: float64

In [43]:
(historical_chip_frauds["mcc_key"] == 10).mean() * 100

np.float64(0.0)

In [44]:
mcc10_fraud_2017_mask = (
    (y_val_2017.to_numpy() == True) &
    (X_val_2017["mcc_key"].to_numpy() == 10)
)

mcc10_fraud_2017 = y_pred_2017[mcc10_fraud_2017_mask]

len(mcc10_fraud_2017)

34

In [45]:
mcc10_fraud_2017.mean() * 100

np.float64(0.0)

##### MCC Distribution Shift

The MCC distribution of Chip Transaction frauds changed substantially in 2017.

Most notably, `mcc_key = 10` accounted for approximately 19% of Chip Transaction frauds in 2017, while no historical Chip Transaction frauds belonged to this category.

The model failed to identify any of the 34 MCC 10 fraud cases present in the 2017 validation fold, resulting in a 0% recall for this subgroup.

This suggests that the poor 2017 performance is associated not only with changes in transaction amounts, but also with a shift in the categorical composition of fraudulent transactions. However, this result does not establish MCC 10 as the direct cause of the prediction failures, since other feature changes may occur simultaneously.

In [46]:
chip_fraud_2017["merchant_id"].value_counts()

merchant_id
83018    23
99370    11
48919    11
86410     7
67570     7
53492     6
41375     6
46284     6
43293     5
59474     5
2297      5
75034     5
80114     5
2156      5
19463     4
31893     4
88260     4
22204     4
7777      3
16790     3
51300     3
90915     3
32070     3
4439      3
66052     2
61195     2
66615     2
83271     2
59935     2
68751     2
1744      2
33191     2
9391      2
11468     2
94625     1
34490     1
14528     1
4906      1
49637     1
17587     1
5059      1
24504     1
45992     1
Name: count, dtype: int64

In [47]:
chip_fraud_2017["merchant_id"].value_counts().value_counts().sort_index()

count
1      9
2     10
3      6
4      4
5      6
6      3
7      2
11     2
23     1
Name: count, dtype: int64

In [48]:
historical_chip_fraud_merchants = historical_chip_frauds["merchant_id"].unique()
historical_chip_fraud_merchants

array([ 68947,  92850,   1356,  69015,  97537,  53076,  92993,  82456,
        45512,  54106,  18422,  34032,  97333,   8635,  90116,  11901,
        25243,  51623,  48136,  85820,  78595,  18786,  46978,  78105,
        61672,  60614,  40594,  71663,  51295,  69161,  69807,  39582,
        95830,  50021,  49501,  61714,  76245,  32868,  97909,  66080,
        40080,   9178,  82549,  78711,    256,  73829,  18987,  29153,
        16443,  17530,  57422,   5802,  25206,  89285,  72650,  79466,
        70364,  45081,  22096,  36440,   7882,  55076,  40815,  87425,
        61530,  76134,  34161,  29657,  38963,  35695,  84477,  10617,
        80817,  36413,  76940,  30068,  63669,  75518,  77786,  85957,
        14945,  16659,  13730,  79447,  43353,   9156,  19115,  54658,
        19557,   5260,  47636,  49670,  47786,  92290,  83316,  60368,
        20494,  97077,  10197,  27442,  24171,  64903,  70082,  15142,
        70625,  73578,   7613,  20303,   5343,  47007,  88985,  44886,
      

In [49]:
known_merchant_2017_mask = chip_fraud_2017["merchant_id"].isin(historical_chip_fraud_merchants)
known_merchant_2017_mask.mean() * 100

np.float64(0.0)

In [50]:
historical_merchants = X_train_2017["merchant_id"].unique()
historical_merchants

array([86369, 75781, 14528, ..., 25390, 83825, 25129], shape=(17635,))

In [51]:
known_historical_merchant_2017_mask = chip_fraud_2017["merchant_id"].isin(historical_merchants)
known_historical_merchant_2017_mask.mean() * 100

np.float64(85.88235294117646)

In [52]:
chip_fraud_2017_merchants = chip_fraud_2017["merchant_id"].unique()
len(chip_fraud_2017_merchants)

43

In [53]:
known_unique_merchants_2017_mask = np.isin(
    chip_fraud_2017_merchants,
    historical_merchants
)
known_unique_merchants_2017_mask.mean() * 100

np.float64(79.06976744186046)

In [54]:
known_merchant_chip_fraud_2017_mask = (
    (y_val_2017.to_numpy() == True) &
    (X_val_2017["use_chip"].to_numpy() == "Chip Transaction") &
    (X_val_2017["merchant_id"].isin(historical_merchants).to_numpy())
)
known_merchant_chip_fraud_2017_mask.sum()

np.int64(146)

In [55]:
known_merchant_chip_fraud_predictions_2017 = y_pred_2017[
    known_merchant_chip_fraud_2017_mask
]
known_merchant_chip_fraud_predictions_2017.mean() * 100

np.float64(0.0)

In [56]:
new_merchant_chip_fraud_2017_mask = (
    (y_val_2017.to_numpy() == True) &
    (X_val_2017["use_chip"].to_numpy() == "Chip Transaction") &
    (~X_val_2017["merchant_id"].isin(historical_merchants).to_numpy())
)

new_merchant_chip_fraud_2017_mask.sum()

np.int64(24)

In [57]:
new_merchant_chip_fraud_predictions_2017 = y_pred_2017[
    new_merchant_chip_fraud_2017_mask
]

new_merchant_chip_fraud_predictions_2017.mean() * 100

np.float64(20.833333333333336)

### Fold 5 Diagnostics Conclusion

The exceptionally low F1-score in the 2017 validation fold is associated with a substantial shift in the transaction and fraud patterns compared with the historical training data.

The most important change is the transition towards Chip Transactions. While Chip Transactions were absent before 2015, they account for almost all fraudulent transactions in 2017.

Further analysis indicates that the characteristics of these frauds also changed. Fraudulent Chip Transactions in 2017 resemble legitimate Chip Transactions more closely in terms of transaction amount and user/card spending behaviour than historical Chip frauds.

Categorical distributions changed as well. New fraud patterns appeared within MCC categories, including MCC 10, which was absent from historical Chip frauds and achieved 0% recall in the 2017 validation fold.

Merchant analysis provides additional evidence that the issue is not simply caused by previously unseen merchants. Most 2017 Chip frauds occurred at merchants already present in the training data, yet the model failed to detect these cases.

Overall, the results suggest temporal concept drift between the historical training data and the 2017 fraud patterns. This explains why the final temporal validation fold performs substantially worse than the earlier folds and highlights the importance of evaluating fraud detection models using time-aware validation.

## Tree Depth Under Temporal Validation

The temporal diagnostics revealed substantial variation in model performance across validation years, including pronounced deterioration in 2015 and 2017 associated with distribution shift.

To verify whether the tree-depth conclusions from the manual experiments hold under time-aware validation, three substantially different depths are compared across the same temporal folds. Each configuration is evaluated both on its mean F1-score and on its stability across years.

In [58]:
max_depths = [2, 6, 10]
all_depths_scores = []

for max_depth in max_depths:
    fold_f1_scores = []

    for (train_indices, _), year in zip(folds, validation_years):
        X_train, y_train, X_val, y_val = prepare_fold(train_indices, year)

        gb = GradientBoostingClassifier(max_depth=max_depth, random_state=42)
        gb.fit(X_train, y_train)

        y_pred = gb.predict(X_val)
        fold_f1_scores.append(f1_score(y_val, y_pred))

    all_depths_scores.append(fold_f1_scores)

all_depths_scores

[[0.19256933542647828,
  0.2847457627118644,
  0.3645859238995275,
  0.44866198519643197,
  0.0],
 [0.5192940472629375,
  0.3733826247689464,
  0.05855773785664833,
  0.6248736097067745,
  0.004570383912248629],
 [0.38376237623762377,
  0.32576210400478184,
  0.042448537911020386,
  0.5297977036632039,
  0.008675342008675343]]

In [59]:
for max_depth, scores in zip(max_depths, all_depths_scores):
    print(f"max_depth={max_depth}: mean F1 = {np.mean(scores):.4f}")

max_depth=2: mean F1 = 0.2581
max_depth=6: mean F1 = 0.3161
max_depth=10: mean F1 = 0.2581


### Initial `max_depth` Evaluation

As an initial step, three substantially different tree depths were evaluated using temporal cross-validation.

When evaluated on full-year validation data, `max_depth=6` achieved the highest mean F1-score (0.316) and outperformed `max_depth=10` (0.258) in four of the five validation years. Deeper trees therefore did not generalize better to future periods, despite their stronger performance in the manual experiments on the 2018 validation set.

The shallow `max_depth=2` model performed notably better in 2015 (F1 0.365 vs 0.059), suggesting that simpler trees were less affected by the shift towards Chip Transactions. However, it failed to detect any fraud in 2017 and performed worse in the remaining years.

Mean F1-scores should be interpreted together with per-year results, as they are strongly influenced by the 2015 and 2017 periods affected by distribution shift.

In [61]:
ap_results = []
fold_scores = {}

for (train_indices, _), year in zip(folds, validation_years):
    X_train, y_train, X_val, y_val = prepare_fold(train_indices, year)

    gb = GradientBoostingClassifier(max_depth=6, random_state=42)
    gb.fit(X_train, y_train)

    scores = gb.predict_proba(X_val)[:, 1]
    fold_scores[year] = scores

    average_precision = average_precision_score(y_val, scores)
    fraud_rate = y_val.mean()

    ap_results.append({
        "year": year,
        "average_precision": average_precision,
        "fraud_rate": fraud_rate,
        "ap_lift": average_precision / fraud_rate
    })

ap_results = pd.DataFrame(ap_results)
ap_results

,year,average_precision,fraud_rate,ap_lift
0,2013,0.355941,0.001474,241.545805
1,2014,0.288896,0.000726,398.133620
2,2015,0.339770,0.002353,144.386412
3,2016,0.532972,0.002624,203.078555
4,2017,0.000710,0.000184,3.868746


## Summary and Next Steps

Key findings:

- **Evaluation at the natural fraud rate is essential.** Undersampling inflates the fraud share of a validation year (e.g. 5.35% instead of 0.24% in 2015), which would overstate precision and F1. On complete yearly data, F1-scores ranged from 0.005 to 0.625.
- **Model performance is not stable over time.** The 2015 degradation is associated with the introduction of Chip Transactions, which led to 51,090 false positives. The 2017 collapse is associated with a change in fraud patterns, including new MCC categories and fraud amounts resembling legitimate transactions.
- **Time-aware validation changed model selection.** Under temporal cross-validation, `max_depth=6` achieved the highest mean F1-score (0.316) and outperformed `max_depth=10` in four of five years, in contrast to the manual experiments on 2018, where deeper trees performed better.

Next steps:

- threshold-independent evaluation (average precision) on the same temporal folds, to separate ranking failures from threshold effects in 2015 and 2017,
- systematic hyperparameter tuning using the same temporal folds,
- selection of the final configuration based on temporal cross-validation,
- a single final evaluation on the 2018 validation set and the unseen 2019 period.